In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import current_timestamp

In [0]:
df_invoice = spark.read.format("delta").table("dev.tracking_br.invoice")

In [0]:
from pyspark.sql.functions import col, to_date
df_invoice = df_invoice.select(col("invno").alias("invoiceno"),
                               col("invdate").alias("invoicedate"),
                               col("ord").alias("orderno"),
                               col("amount").alias("invoiceamount"),
                               col("taxrate"),
                               col("taxamount"),
                               col("paymentdays"),
                               col("nameofcustomer"),
                               col("city").alias("invoicecity"),
                               col("state").alias("invoicestate"))

                               

In [0]:
from pyspark.sql.functions import to_timestamp, col
df_invoice = convert_to_int(df_invoice, "paymentdays")
df_invoice = df_invoice.withColumn("ingestion_time",current_timestamp())
df_invoice = convert_to_timestamp(df_invoice, "invoicedate")
df_invoice= df_invoice.withColumn("invoicedate", to_timestamp(col("invoicedate")))

In [0]:
target_columns = ["invoiceamount", "taxrate","taxamount"]

# Apply the UDF in a loop
for col_name in target_columns:
     df_invoice = convert_to_float(df_invoice, col_name)


In [0]:
df_invoice.createOrReplaceTempView("temp_invoice")

In [0]:
%sql
MERGE INTO dev.tracking_sl.invoice tgt 
USING ( 
  SELECT DISTINCT * 
  FROM temp_invoice 
) src 
ON tgt.invoiceno = src.invoiceno 
WHEN NOT MATCHED THEN 
  INSERT *